In [ ]:
# # UnicornForge AI — AMD Training Notebook
# Canonical training flow for `SuccessScoreMLP` on `global_startup_success_dataset.csv`.
# For reproducible training from CLI, use: `python train_model.py`


In [ ]:
import torch
import pandas as pd

from ml.dataset import DEFAULT_DATASET_PATH, get_dataset_info
from ml.prompts import build_unicornforge_prompt, row_to_description
from ml.training import train_success_model

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# ## 1. Load dataset


In [ ]:
dataset_info = get_dataset_info()
print(dataset_info)

df = pd.read_csv(DEFAULT_DATASET_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
# ## 2. Build startup descriptions and prompts (used by brief generation)


In [ ]:
df["startup_description"] = df.apply(row_to_description, axis=1)
df["uf_prompt"] = df["startup_description"].apply(build_unicornforge_prompt)
df[["Startup Name", "startup_description", "Success Score"]].head(5)


In [ ]:
# ## 3. Train SuccessScoreMLP and export artifacts for backend inference


In [ ]:
result = train_success_model(epochs=20)
result


In [ ]:
# ## 4. Verify backend can load the exported model


In [ ]:
from ml.predictor import SuccessPredictor

predictor = SuccessPredictor()
print("Predictor ready:", predictor.ready)
print("Feature columns:", len(predictor.feature_columns))

sample = predictor.predict_from_payload(
    project_idea="AI tutor for university students",
    industry="EdTech",
    available_technologies="Python, AMD GPUs",
    available_time="48 hours",
)
print("Sample score:", sample.score if sample else None, sample.label if sample else None)


In [ ]:
# ## 5. Evaluate model quality (MAE / RMSE / R²)


In [ ]:
from ml.evaluation import evaluate_saved_model

metrics = evaluate_saved_model()
print("Validation metrics:", metrics["metrics"])
print("Sample predictions:", metrics["predictions"])


In [ ]:
# Optional: plot training loss curve if history is stored in metadata.


In [ ]:
import pickle
from pathlib import Path

metadata_path = Path("trained_models/startup_success_mlp/metadata.pkl")
if metadata_path.exists():
    with open(metadata_path, "rb") as handle:
        metadata = pickle.load(handle)
    history = metadata.get("history", [])
    if history:
        import matplotlib.pyplot as plt

        epochs = [item["epoch"] for item in history]
        train_mse = [item["train_mse"] for item in history]
        val_mse = [item["val_mse"] for item in history]
        plt.figure(figsize=(8, 4))
        plt.plot(epochs, train_mse, label="train MSE")
        plt.plot(epochs, val_mse, label="val MSE")
        plt.xlabel("Epoch")
        plt.ylabel("MSE")
        plt.title("SuccessScoreMLP training curve")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No training history in metadata. Retrain with: python train_model.py")
else:
    print("Metadata not found. Train the model first.")